In [1]:
# import numpy as np
import cupy as cp
import time
import random
import copy
from enum import Enum, auto
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

In [1]:
import cudaq
cudaq.set_target("qpp")

k = cudaq.make_kernel()
print("qbit attr:", k.qbit)
print("qbit type:", type(k.qbit))

qs = k.qbit(3)
print("allocated qubits:", qs)


RuntimeError: Invalid target name (qpp).

In [2]:
class GateType(Enum):
    X = auto()
    Y = auto()
    Z = auto()
    H = auto()

GATE_UNITARIES: Dict[GateType, List[List[complex]]]= {
    GateType.X: [[0, 1],
                [1, 0]],
    GateType.Y: [[0, -1j],
                [1j, 0]],
    GateType.Z: [[1, 0],
                [0, -1]],
    GateType.H: [[1/2**0.5, 1/2**0.5],
                [1/2**0.5, -1/2**0.5]],
}

NUM_QUBITS = 3
NUM_OPERATIONS = 10000
MATRIX_SIZE = 2 * NUM_QUBITS + 1

@dataclass
class SingleQubitGate:
    gate: GateType
    a: complex
    b: complex
    c: complex
    d: complex
    target: int

@dataclass
class ControlledGate:
    gate: GateType
    a: complex
    b: complex
    c: complex
    d: complex
    target: int
    control: List[int]

GateOperation = SingleQubitGate | ControlledGate

def customize(gate: GateType, target: int, control: List[int] | None = None) -> GateOperation:
    matrix = GATE_UNITARIES[gate]
    a, b = matrix[0]
    c, d = matrix[1]
    if control == None:
        return SingleQubitGate(gate, a, b, c, d, target)
    else:
        return ControlledGate(gate, a, b, c, d, target, control)


def randomizer(option: Optional[GateOperation], threshold=0.5) -> List[GateOperation]:
    operations: List[GateOperation] = []

    for _ in range(NUM_OPERATIONS):
        gate = random.choice(list(GateType))
        matrix = GATE_UNITARIES[gate]
        a, b = matrix[0]
        c, d = matrix[1]
        
        if option == SingleQubitGate or random.random() < threshold:
            target = random.randint(0, NUM_QUBITS - 1)
            operations.append(SingleQubitGate(gate, a, b, c, d, target))
        elif option == ControlledGate or random.random() >= threshold:
            control, target = random.sample(range(NUM_QUBITS), 2)
            operations.append(ControlledGate(gate, a, b, c, d, target, [control]))
    
    return operations

def initialization(L, R) -> Tuple[List[List[complex]], List[List[complex]], List[List[int]]]:    
    if 1 < MATRIX_SIZE:
        L[1, 0] = 1  

    for col in range(1, MATRIX_SIZE - 2):  
        row = col + 2
        if row < MATRIX_SIZE:
            L[row, col] = 1

    if 2 < MATRIX_SIZE:
        R[2, 0] = 1  

    for col in range(1, MATRIX_SIZE - 2):  
        if col % 2 == 1:  
            row = col + 3
        else:  
            row = col + 2
        if row < MATRIX_SIZE:
            R[row, col] = 1

    Mapping = []
    Mapping.append([0])

    for i in range(1, NUM_QUBITS):
        Mapping.append([2*i - 1, 2*i])
    Mapping.append([2*i + 1])
    Mapping.append([2*i + 2])

    return L, R, Mapping


def _view_operations(operations: List[GateOperation]):
    for op in operations:
        print(op.gate.name, op.target)

In [19]:
# operations = randomizer(option=SingleQubitGate)

operations : List[GateOperation] = []
operations.append(customize(GateType.X, 0))
operations.append(customize(GateType.X, 1))
operations.append(customize(GateType.X, 2))
operations.append(customize(GateType.H, 0))
operations.append(customize(GateType.H, 1))
operations.append(customize(GateType.H, 2))

# operations.append(customize(GateType.X, 2))
# operations.append(customize(GateType.Z, 2))

# _view_operations(operations)

In [20]:
import cupy as cp

def compute_result(L: cp.ndarray, R: cp.ndarray, seq: List[int], leaf: List[int]) -> complex:
    A = cp.zeros((1, MATRIX_SIZE))
    for i in leaf:
        A[0, i] = 1

    B = cp.zeros((MATRIX_SIZE, 1))
    B[0, 0] = 1

    dirs = B
    for dir in seq:
        if dir == 0:
            dirs = L @ dirs
        else:
            dirs = R @ dirs
        
    result = A @ dirs

    return result.item()

In [5]:
cp_start = time.time()

In [21]:
L = cp.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
R = cp.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)

L, R, Mapping = initialization(L, R)
# print(L)
# print(R)
# print(Mapping)

In [ ]:
for op in operations:
    L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)
    
    A = cp.eye(MATRIX_SIZE, dtype=complex)
    B = cp.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
    C = cp.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
    D = cp.eye(MATRIX_SIZE, dtype=complex)
    
    for node in Mapping[op.target]:
        A[node][node] = op.a
        B[node][node] = op.b
        C[node][node] = op.c
        D[node][node] = op.d

    # L_U = cp.sqrt(cp.square(L @ A) + cp.square(R @ B))
    # R_U = cp.sqrt(cp.square(L @ C) + cp.square(R @ D))
    L_U = L @ A + R @ B
    R_U = L @ C + R @ D

    if isinstance(op, SingleQubitGate):
        L, R = L_U, R_U

    if not L.shape == R.shape:
        raise ValueError("Size of L and R not equal")
    
    MATRIX_SIZE = L.shape[0]

In [ ]:
# for op in operations:
#     L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)
    
#     A = cp.eye(MATRIX_SIZE, dtype=complex)
#     B = cp.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
#     C = cp.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
#     D = cp.eye(MATRIX_SIZE, dtype=complex)
    
#     for node in Mapping[op.target]:
#         A[node][node] = op.a
#         B[node][node] = op.b
#         C[node][node] = op.c
#         D[node][node] = op.d

#     L_U = L @ A + R @ B
#     R_U = L @ C + R @ D

#     print("1: \n", R)

#     if isinstance(op, SingleQubitGate):
#         L, R = L_U, R_U
    
#     elif isinstance(op, ControlledGate):
#         L_top = L[:MATRIX_SIZE-2, :MATRIX_SIZE-2]
#         L_bottom = L[MATRIX_SIZE-2:, :MATRIX_SIZE-2]

#         L_U_top = L_U[:MATRIX_SIZE-2, :MATRIX_SIZE-2]
#         L_U_bottom = L_U[MATRIX_SIZE-2:, :MATRIX_SIZE-2]

#         top = cp.block([
#             [L_top, cp.zeros((MATRIX_SIZE-2, MATRIX_SIZE-2), dtype=complex)],
#             [cp.zeros((MATRIX_SIZE-2, MATRIX_SIZE-2), dtype=complex), L_U_top]
#         ])

#         top = cp.hstack([top, cp.zeros(((MATRIX_SIZE-2)*2, 2), dtype=complex)])

#         bottom = cp.hstack([L_bottom, L_U_bottom, cp.zeros((2, 2), dtype=complex)])

#         L = cp.vstack([top, bottom])

#         R_top = R[:MATRIX_SIZE-2, :MATRIX_SIZE-2]
#         R_bottom = R[MATRIX_SIZE-2:, :MATRIX_SIZE-2]

#         print("2: \n", R_top)

#         R_U_top = R_U[:MATRIX_SIZE-2, :MATRIX_SIZE-2]
#         R_U_bottom = R_U[MATRIX_SIZE-2:, :MATRIX_SIZE-2]

#         print("3: \n", R_top)

#         top = cp.block([
#             [R_top, cp.zeros((MATRIX_SIZE-2, MATRIX_SIZE-2), dtype=complex)],
#             [cp.zeros((MATRIX_SIZE-2, MATRIX_SIZE-2), dtype=complex), R_U_top]
#         ])

#         top = cp.hstack([top, cp.zeros(((MATRIX_SIZE-2)*2, 2), dtype=complex)])

#         bottom = cp.hstack([R_bottom, R_U_bottom, cp.zeros((2, 2), dtype=complex)])

#         R = cp.vstack([top, bottom])
#         print(R)
#         if len(op.control) == 1:
#             control = op.control[0]
#             print(Mapping[control])
#             for node in Mapping[control]:
#                 for i, amp in enumerate(R[:, node]):
#                     if not amp == 0:
#                         print(i)
#                         R[:, i] = R[:, i + MATRIX_SIZE - 2]
                
#             for mapping in Mapping[:-2]:
#                 mapping_U = []
#                 for node in mapping:
#                     mapping_U.append(node + MATRIX_SIZE - 2)
#                 mapping += mapping_U
#             Mapping[-2][0] += MATRIX_SIZE - 2
#             Mapping[-1][0] += MATRIX_SIZE - 2
                
#         else:
#             for control in op.control:
#                 for node in Mapping[control]:
#                     ...
                    
#     if not L.shape == R.shape:
#         raise ValueError("Size of L and R not equal")
    
#     MATRIX_SIZE = L.shape[0]

In [8]:
cp_end = time.time()


In [23]:
print(compute_result(L, R, [0] * NUM_QUBITS, Mapping[3]), compute_result(L, R, [0, 0, 1], Mapping[3]), compute_result(L, R, [0, 1, 0], Mapping[3]), compute_result(L, R, [0, 1, 1], Mapping[3]))

(0.3535533905932737+0j) (-0.3535533905932737+0j) (-0.3535533905932737+0j) (0.3535533905932737+0j)


In [10]:
import numpy as np

def compute_result(L: np.ndarray, R: np.ndarray, seq: List[int], leaf: List[int]) -> complex:
    A = np.zeros((1, MATRIX_SIZE))
    for i in leaf:
        A[0, i] = 1

    B = np.zeros((MATRIX_SIZE, 1))
    B[0, 0] = 1

    dirs = B
    for dir in seq:
        if dir == 0:
            dirs = L @ dirs
        else:
            dirs = R @ dirs
        
    result = A @ dirs

    return result.item()

In [11]:
np_start = time.time()

In [12]:
L = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
R = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)

L, R, Mapping = initialization(L, R)
# print(L)
# print(R)
# print(Mapping)


In [13]:
for op in operations:
    L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)
    
    A = np.eye(MATRIX_SIZE, dtype=complex)
    B = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
    C = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=complex)
    D = np.eye(MATRIX_SIZE, dtype=complex)
    
    for node in Mapping[op.target]:
        A[node][node] = op.a
        B[node][node] = op.b
        C[node][node] = op.c
        D[node][node] = op.d

    L_U = np.sqrt(np.square(L @ A) + np.square(R @ B))
    R_U = np.sqrt(np.square(L @ C) + np.square(R @ D))

    if isinstance(op, SingleQubitGate):
        L, R = L_U, R_U

    if not L.shape == R.shape:
        raise ValueError("Size of L and R not equal")
    
    MATRIX_SIZE = L.shape[0]
    
    # _view_operations([op])
    # print(L)
    # print(R)
    # print(Mapping)


In [14]:
np_end = time.time()


In [15]:
compute_result(L, R, [0] * NUM_QUBITS, Mapping[3])


(0.3535533905932737+0j)

In [16]:
print(cp_end - cp_start)
print(np_end - np_start)

0.3042116165161133
0.027769088745117188
